In [49]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [51]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
    ]


In [52]:
query = f"""DROP VIEW IF EXISTS Yearly_KPI;"""
cursor.execute(query)
conn.commit()


In [53]:
query = """
CREATE VIEW IF NOT EXISTS Yearly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [54]:
print(query)


CREATE VIEW IF NOT EXISTS Yearly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [55]:
query = "SELECT * FROM Yearly_KPI WHERE Year = 2025;"
df = pd.read_sql_query(query, conn)
conn.close()
df

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,B1Density,Bm1Density,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2025,None,ITALGAS,None,174739.59,163152.36,174739.59,163152.36,None,0.0,...,0.01,0.79,0.22,13.36,0.45,67.35,18.83,43.13,32.97,23.91
1,2025,None,ITALGAS,Abruzzo,7112.37,6626.05,7112.37,6626.05,None,0.0,...,0.00,0.72,0.18,13.85,0.43,68.66,17.06,58.09,24.09,17.82
2,2025,None,ITALGAS,Calabria,11554.36,10535.42,11554.36,10535.42,None,0.0,...,0.00,0.47,0.10,17.19,0.53,68.19,14.09,46.90,35.33,17.77
3,2025,None,ITALGAS,Campania,11566.76,10840.63,11566.76,10840.63,None,0.0,...,0.01,1.26,0.38,10.87,0.38,67.97,20.77,46.27,33.60,20.14
4,2025,None,ITALGAS,Centro Italia,11355.49,10503.80,11355.49,10503.80,None,0.0,...,0.01,0.70,0.16,16.87,0.84,66.61,15.67,51.92,28.39,19.69
5,2025,None,ITALGAS,Lazio Sud,8707.53,8149.39,8707.53,8149.39,None,0.0,...,0.00,0.73,0.19,12.26,0.38,69.17,18.19,58.07,19.87,22.07
6,2025,None,ITALGAS,Liguria,8493.07,7828.69,8493.07,7828.69,None,0.0,...,0.00,0.64,0.15,14.46,0.46,69.19,15.89,48.67,36.36,14.98
7,2025,None,ITALGAS,Lombardia Nord-Est,5884.65,5548.85,5884.65,5548.85,None,0.0,...,0.00,0.93,0.26,10.09,0.34,70.23,19.34,32.27,31.57,36.16
8,2025,None,ITALGAS,Lombardia Nord-Ovest,5880.10,5466.82,5880.10,5466.82,None,0.0,...,0.00,0.79,0.30,7.42,0.18,66.85,25.55,39.25,33.11,27.64
9,2025,None,ITALGAS,Lombardia Sud,7936.09,7099.84,7936.09,7099.84,None,0.0,...,0.01,0.92,0.26,13.61,0.70,66.79,18.90,37.12,26.43,36.45


In [56]:
Query(query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'ITALGAS')").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_ITALGAS_Y2023_W2,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Week,2.0,None,None,2026-07-21 18:46:13.907648
1,FOVMain_ITALGAS_Y2023_W3,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Week,3.0,None,None,2026-07-21 18:46:13.907648
2,FOVMain_ITALGAS_Y2023_W4,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Week,4.0,None,None,2026-07-21 18:46:13.907648
3,FOVMain_ITALGAS_Y2023_W5,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Week,5.0,None,None,2026-07-21 18:46:13.907648
4,FOVMain_ITALGAS_Y2023_W6,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Week,6.0,None,None,2026-07-21 18:46:13.907648
...,...,...,...,...,...,...,...,...,...,...
77579,PGShare_ITALGAS_Y2026,PGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2026,Year,NaN,31.38,None,2026-07-21 18:46:30.700236
77580,Not_NGShare_ITALGAS_Y2023,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2023,Year,NaN,18.67,None,2026-07-21 18:46:30.700236
77581,Not_NGShare_ITALGAS_Y2024,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2024,Year,NaN,22.99,None,2026-07-21 18:46:30.700236
77582,Not_NGShare_ITALGAS_Y2025,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Year,NaN,23.91,None,2026-07-21 18:46:30.700236
